# Simplex method benchmarks

In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

import sys
# sys.path.append('.')
# sys.path.append('./DynaMix/')

plt.rcParams["font.family"] = "Helvetica"

In [ ]:
all_k

array([[2.21351391, 2.19561983],
       [2.02563069, 2.13638088]])

In [18]:
from scipy.stats import pearsonr, spearmanr

model_names = ["parrot", "dynamix", "Chronos", "simplex"]
# model_names = ["parrot", "dynamix",  "simplex"]

from scipy.stats import norm
def err_from_corr(corr, pvalue):
    z_obs = np.arctanh(corr)
    z_stat = abs(norm.ppf(pvalue / 2))
    sd_z = abs(z_obs) / z_stat if z_stat != 0 else np.nan
    sd_r = sd_z * (1 - corr**2)
    return sd_r

for model_name in model_names:
    results_path = f"../analysis/{model_name}_statistics/"
    all_lyap = np.load(results_path + '/all_lyap_longhorizon.npy', allow_pickle=True)
    all_kl_dist = np.load(results_path + '/all_kl_dist_longhorizon.npy', allow_pickle=True)
    all_cdim = np.load(results_path + '/all_cdim_longhorizon.npy', allow_pickle=True)

    print(model_name)
    print(f"KL: {np.mean(all_kl_dist)}, {np.std(all_kl_dist) / np.sqrt(len(all_kl_dist))}")
    print(spearmanr(np.array(all_cdim)[:,0], np.array(all_cdim)[:,1])[0], err_from_corr(spearmanr(np.array(all_cdim)[:,0], np.array(all_cdim)[:,1])[0], spearmanr(np.array(all_cdim)[:,0], np.array(all_cdim)[:,1])[1]))
    print(spearmanr(np.array(all_lyap)[:,0], np.array(all_lyap)[:,1])[0], err_from_corr(spearmanr(np.array(all_lyap)[:,0], np.array(all_lyap)[:,1])[0], spearmanr(np.array(all_lyap)[:,0], np.array(all_lyap)[:,1])[1]))
    # print(spearmanr(np.array(all_cdim)[:,0], np.array(all_cdim)[:,1]))
    # print(spearmanr(np.array(all_lyap)[:, 0], np.array(all_lyap)[:, 1]))
    print("\n")



parrot
KL: 0.4770757422644982, 0.19669104527779788
0.7327242046343169 0.052727576735238466
0.10832802838698302 0.10575277429678523


dynamix
KL: 0.38488280816013243, 0.10033072569154951
0.6061355311355311 0.08376235385460877
0.34148351648351644 0.11380642884050807


Chronos
KL: 0.4890113277645498, 0.18906586915921275
0.9999999999999999 nan
-0.9999999999999999 nan


simplex
KL: 0.43807683354707644, 0.10110958639162314
0.3602780086122233 0.11411726729761233
0.2695613401911192 0.12097246736268179




# Lyapunov exponent calculation

In [2]:
import sys
## add the path to the project
sys.path.append('..')

from models.parrot import SimplexForecaster, context_parroting_forecast


In [2]:


# ?max_lyapunov_exponent_rosenstein
import sys
## add the path to the project
sys.path.append('..')

from models.parrot import SimplexForecaster


In [22]:
import torch
from chronos import BaseChronosPipeline, ChronosPipeline, ChronosBoltPipeline

pipeline = BaseChronosPipeline.from_pretrained(
    "amazon/chronos-t5-base",  # use "amazon/chronos-bolt-small" for the corresponding Chronos-Bolt model
    #"amazon/chronos-bolt-base",
    #device_map="cuda",  # use "cpu" for CPU inference
    device_map="cpu",
    torch_dtype=torch.bfloat16,
)

In [3]:
import sys
# sys.path.append('.')
sys.path.append('./DynaMix/')

import torch
# from src.model.dynamix import DynaMix
from src.model.forecaster import DynaMixForecaster
# from src.metrics.metrics import geometrical_misalignment, temporal_misalignment, MASE
# from src.utilities.plotting_eval import plot_3D_attractor, plot_2D_attractor, plot_TS_forecast
from src.utilities.utilities import load_hf_model

# Load the pre-trained model
model = load_hf_model("dynamix-3d-alrnn-v1.0")
model.eval() # Set model to evaluation mode
forecaster = DynaMixForecaster(model) # Initialize the forecaster

/Users/william/program_repos/parroting/benchmark/DynaMix/src/model/dynamix.py:95: RuntimeWarning: divide by zero encountered in matmul
  K = R.T @ R / M + np.eye(M)
/Users/william/program_repos/parroting/benchmark/DynaMix/src/model/dynamix.py:95: RuntimeWarning: overflow encountered in matmul
  K = R.T @ R / M + np.eye(M)
/Users/william/program_repos/parroting/benchmark/DynaMix/src/model/dynamix.py:95: RuntimeWarning: invalid value encountered in matmul
  K = R.T @ R / M + np.eye(M)


In [ ]:
import glob
from dysts.analysis import max_lyapunov_exponent_rosenstein, max_lyapunov_exponent_rosenstein_multivariate, gp_dim
from dysts.metrics import estimate_kl_divergence


results_path = "../analysis/simplex_statistics/"
CHECK_EXISTING = False

context_length = 512
forecast_length = 10000 - context_length


if CHECK_EXISTING:
    all_lyap = np.load(results_path + '/all_lyap_longhorizon.npy', allow_pickle=True).tolist()
    all_cdim = np.load(results_path + '/all_cdim_longhorizon.npy', allow_pickle=True).tolist()
    all_kl_dist = np.load(results_path + '/all_kl_dist_longhorizon.npy', allow_pickle=True).tolist()
else:
    all_lyap = list()
    all_cdim = list()
    all_kl_dist = list()

for trajectory in sorted(glob.glob("../data/long_trajectories/*.npy")):
    equation_name = trajectory.split("/")[-1].split(".")[0]
    print(equation_name, flush=True)
    # if equation_name in found_systems:
    #     print(f"Skipping {equation_name} because it's already in the found_systems list", flush=True)
    #     continue
    
    traj = np.load(trajectory, allow_pickle=True)
    traj = (traj - np.mean(traj, axis=0)) / np.std(traj, axis=0)

    try:

        traj_context = traj[:context_length, :]
        traj_true = traj[context_length:context_length+forecast_length, :]



        # traj_pred = list()
        # for mode in range(traj_context.shape[1]):
        #     traj_pred_mode = context_parroting_forecast(traj_context[:, mode], forecast_total_length=forecast_length)[2]
        #     traj_pred.append(traj_pred_mode)
        # traj_pred = np.array(traj_pred).T

        traj_pred = list()
        for mode in range(traj_context.shape[1]):
            model = SimplexForecaster()
            model.fit(traj_context[:, mode])
            traj_pred_mode = model.forecast(forecast_length)
            traj_pred.append(traj_pred_mode)
        traj_pred = np.array(traj_pred).T


        # traj_pred = list()
        # for mode in range(traj_context.shape[1]):
        #     forecast = pipeline.predict(
        #         inputs=torch.tensor(traj_context[:, mode]),
        #         prediction_length=forecast_length,
        #         num_samples=20,
        #         limit_prediction_length=False,
        #         )
        #     traj_pred_mode = np.mean(forecast[0, :, :].detach().numpy(), axis=0)
        #     traj_pred.append(traj_pred_mode)
        # traj_pred = np.array(traj_pred).T


        # context_traj_tensor  = torch.tensor(traj_context)
        # with torch.no_grad(): 
        #     reconstruction = forecaster.forecast(
        #         context=context_traj_tensor,
        #         horizon=forecast_length, # Match the horizon of the ground truth
        #         standardize=True,
        #     )
        # traj_pred = reconstruction.detach().numpy()

        
        kl_dist = estimate_kl_divergence(traj_true, traj_pred)
        if np.isinf(kl_dist): kl_dist = np.nan
        
        
        cdim_true = gp_dim(traj_true)
        cdim_pred = gp_dim(traj_pred)
        cdim = np.array([cdim_pred, cdim_true])

        lyap_true = max_lyapunov_exponent_rosenstein(traj_true)
        if np.isinf(lyap_true): lyap_true = 0
        lyap_pred = max_lyapunov_exponent_rosenstein(traj_pred)
        if np.isinf(lyap_pred): lyap_pred = 0
        lyap = np.array([lyap_pred, lyap_true])

        all_cdim.append(cdim)
        all_kl_dist.append(kl_dist)
        all_lyap.append(lyap)

        np.array(all_cdim).dump(results_path + '/all_cdim_longhorizon.npy')
        np.array(all_kl_dist).dump(results_path + '/all_kl_dist_longhorizon.npy')
        np.array(all_lyap).dump(results_path + '/all_lyap_longhorizon.npy')


    except Exception as e:
        print(e)
        print(f"Skipping {equation_name}", flush=True)
        continue


Aizawa


/Users/william/program_repos/parroting/.venv/lib/python3.13/site-packages/scipy/stats/_covariance.py:633: RuntimeWarning: divide by zero encountered in matmul
  return x @ self._LP
/Users/william/program_repos/parroting/.venv/lib/python3.13/site-packages/scipy/stats/_covariance.py:633: RuntimeWarning: overflow encountered in matmul
  return x @ self._LP
/Users/william/program_repos/parroting/.venv/lib/python3.13/site-packages/scipy/stats/_covariance.py:633: RuntimeWarning: invalid value encountered in matmul
  return x @ self._LP


AnishchenkoAstakhov
Arneodo


In [29]:
from scipy.stats import pearsonr, spearmanr
print(pearsonr(np.array(all_lyap[:10])[:,0], np.array(all_lyap[:10])[:,1]))

PearsonRResult(statistic=np.float64(-0.25356030822853126), pvalue=np.float64(0.4796431061947596))


In [27]:
from scipy.stats import pearsonr, spearmanr
print(pearsonr(np.array(all_lyap[:10])[:,0], np.array(all_lyap[:10])[:,1]))

PearsonRResult(statistic=np.float64(0.18220036987501692), pvalue=np.float64(0.6144063306278694))


In [21]:
from scipy.stats import pearsonr, spearmanr
print(f"KL: {np.mean(all_kl_dist)}, {np.std(all_kl_dist)}")
print(pearsonr(np.array(all_cdim)[:,0], np.array(all_cdim)[:,1]))
print(pearsonr(np.array(all_lyap)[:,0], np.array(all_lyap)[:,1]))
print(spearmanr(np.array(all_cdim)[:,0], np.array(all_cdim)[:,1]))
print(spearmanr(np.array(all_lyap)[:, 0], np.array(all_lyap)[:, 1]))
print("\n")

KL: 0.41572513705957026, 1.6238033699849883
PearsonRResult(statistic=np.float64(0.7229814973970862), pvalue=np.float64(1.8433805897964297e-22))
PearsonRResult(statistic=np.float64(0.14518709741583316), pvalue=np.float64(0.09800664945240242))
SignificanceResult(statistic=np.float64(0.7413761810708376), pvalue=np.float64(4.244057657913758e-24))
SignificanceResult(statistic=np.float64(0.029186974128039396), pvalue=np.float64(0.7406980941180865))




In [5]:
from scipy.stats import pearsonr, spearmanr
print(f"KL: {np.mean(all_kl_dist)}, {np.std(all_kl_dist)}")
print(pearsonr(np.array(all_cdim)[:,0], np.array(all_cdim)[:,1]))
print(pearsonr(np.array(all_lyap)[:,0], np.array(all_lyap)[:,1]))
print(spearmanr(np.array(all_cdim)[:,0], np.array(all_cdim)[:,1]))
print(spearmanr(np.array(all_lyap)[:, 0], np.array(all_lyap)[:, 1]))
print("\n")

KL: 0.17005226602916967, 0.3393070459858422
PearsonRResult(statistic=np.float64(0.6042792341388963), pvalue=np.float64(8.711187497315747e-06))
PearsonRResult(statistic=np.float64(nan), pvalue=np.float64(nan))
SignificanceResult(statistic=np.float64(0.5363552266419981), pvalue=np.float64(0.00012196924852145725))
SignificanceResult(statistic=np.float64(0.04889310472556099), pvalue=np.float64(0.7469405031438301))




/Users/william/program_repos/parroting/.venv/lib/python3.13/site-packages/scipy/stats/_stats_py.py:4750: RuntimeWarning: invalid value encountered in subtract
  xm = x - xmean


In [ ]:
results_path = "../analysis/parrot_statistics/"
np.array(all_cdim).dump(results_path + '/all_cdim_longhorizon.npy')
np.array(all_lyap).dump(results_path + '/all_lyap_longhorizon.npy')
np.array(all_kl_dist).dump(results_path + '/all_kl_dist_longhorizon.npy')

In [38]:
# print(model_name)
print(f"KL: {np.mean(all_kl_dist)}, {np.std(all_kl_dist)}")
print(pearsonr(np.array(all_cdim)[:,0], np.array(all_cdim)[:,1]))
print(pearsonr(np.array(all_lyap)[:,0], np.array(all_lyap)[:,1]))
print(spearmanr(np.array(all_cdim)[:,0], np.array(all_cdim)[:,1]))
print(spearmanr(np.array(all_lyap)[:, 0], np.array(all_lyap)[:, 1]))
print("\n")

KL: 0.4088481413000285, 1.6287799243572592
PearsonRResult(statistic=np.float64(0.7229814973970862), pvalue=np.float64(1.8433805897964297e-22))
PearsonRResult(statistic=np.float64(nan), pvalue=np.float64(nan))
SignificanceResult(statistic=np.float64(0.7413761810708376), pvalue=np.float64(4.244057657913758e-24))
SignificanceResult(statistic=np.float64(0.029186974128039396), pvalue=np.float64(0.7406980941180865))




/Users/william/program_repos/parroting/.venv/lib/python3.13/site-packages/scipy/stats/_stats_py.py:4750: RuntimeWarning: invalid value encountered in subtract
  xm = x - xmean


In [41]:
all_lyap

array([ 4.04817890e-02,  1.17889041e-03,  7.25336913e-01,  1.54648717e-01,
        2.82454568e-01,  4.80976238e-02,  2.71374669e-03,  1.19792249e-01,
        7.44031905e-01, -9.02592298e-04,  2.44704709e-01,  1.50445234e-01,
        5.58529979e-02,  1.05814997e+00,  3.63509047e-02,  1.68053570e-01,
        1.55236857e-01,  2.14587380e-01,  1.06015220e-03, -2.05029139e-04,
        4.81865137e-04,  3.85134575e-02,  2.18019428e-02,  4.63277993e-02,
        2.40281480e-02,  4.93242169e-02,  8.50720425e-02,  8.16807708e-01,
        6.25593062e-03,  2.71080249e-02,  1.12166901e-02,  1.73342685e+00,
        3.12082438e-04,  4.16434688e-02,  6.31355881e-02,  2.18270603e-02,
        2.13037869e-03,  1.02238125e-01,  1.74456011e-01,  8.70137662e-02,
       -1.12149349e-03,  2.23880787e-02,  1.75271105e-01,  4.37015147e-02,
        6.67577239e-02,  2.41791993e-02,  6.75127858e-01,  5.42058923e-02,
        3.28843967e-01,  3.77641437e-01,  1.54776945e-01,  1.04546907e+00,
        1.59795627e-01,  

In [42]:
all_lyap

array([ 4.04817890e-02,  1.17889041e-03,  7.25336913e-01,  1.54648717e-01,
        2.82454568e-01,  4.80976238e-02,  2.71374669e-03,  1.19792249e-01,
        7.44031905e-01, -9.02592298e-04,  2.44704709e-01,  1.50445234e-01,
        5.58529979e-02,  1.05814997e+00,  3.63509047e-02,  1.68053570e-01,
        1.55236857e-01,  2.14587380e-01,  1.06015220e-03, -2.05029139e-04,
        4.81865137e-04,  3.85134575e-02,  2.18019428e-02,  4.63277993e-02,
        2.40281480e-02,  4.93242169e-02,  8.50720425e-02,  8.16807708e-01,
        6.25593062e-03,  2.71080249e-02,  1.12166901e-02,  1.73342685e+00,
        3.12082438e-04,  4.16434688e-02,  6.31355881e-02,  2.18270603e-02,
        2.13037869e-03,  1.02238125e-01,  1.74456011e-01,  8.70137662e-02,
       -1.12149349e-03,  2.23880787e-02,  1.75271105e-01,  4.37015147e-02,
        6.67577239e-02,  2.41791993e-02,  6.75127858e-01,  5.42058923e-02,
        3.28843967e-01,  3.77641437e-01,  1.54776945e-01,  1.04546907e+00,
        1.59795627e-01,  

In [44]:
all_lyap = np.load(results_path + '/all_lyap_longhorizon.npy', allow_pickle=True)
all_lyap

array([ 4.04817890e-02,  1.17889041e-03,  7.25336913e-01,  1.54648717e-01,
        2.82454568e-01,  4.80976238e-02,  2.71374669e-03,  1.19792249e-01,
        7.44031905e-01, -9.02592298e-04,  2.44704709e-01,  1.50445234e-01,
        5.58529979e-02,  1.05814997e+00,  3.63509047e-02,  1.68053570e-01,
        1.55236857e-01,  2.14587380e-01,  1.06015220e-03, -2.05029139e-04,
        4.81865137e-04,  3.85134575e-02,  2.18019428e-02,  4.63277993e-02,
        2.40281480e-02,  4.93242169e-02,  8.50720425e-02,  8.16807708e-01,
        6.25593062e-03,  2.71080249e-02,  1.12166901e-02,  1.73342685e+00,
        3.12082438e-04,  4.16434688e-02,  6.31355881e-02,  2.18270603e-02,
        2.13037869e-03,  1.02238125e-01,  1.74456011e-01,  8.70137662e-02,
       -1.12149349e-03,  2.23880787e-02,  1.75271105e-01,  4.37015147e-02,
        6.67577239e-02,  2.41791993e-02,  6.75127858e-01,  5.42058923e-02,
        3.28843967e-01,  3.77641437e-01,  1.54776945e-01,  1.04546907e+00,
        1.59795627e-01,  

In [30]:
from scipy.stats import pearsonr, spearmanr

model_names = ["parrot", "dynamix", "Chronos", "simplex"]
model_names = ["parrot", "Chronos"]

for model_name in model_names:
    results_path = f"../analysis/{model_name}_statistics/"
    all_lyap = np.load(results_path + '/all_lyap_longhorizon.npy', allow_pickle=True)
    all_kl_dist = np.load(results_path + '/all_kl_dist_longhorizon.npy', allow_pickle=True)
    all_cdim = np.load(results_path + '/all_cdim_longhorizon.npy', allow_pickle=True)

    print(model_name)
    print(f"KL: {np.mean(all_kl_dist)}, {np.std(all_kl_dist) / np.sqrt(len(all_kl_dist))}")
    print(pearsonr(np.array(all_cdim)[:,0], np.array(all_cdim)[:,1]))
    print(pearsonr(np.array(all_lyap)[:,0], np.array(all_lyap)[:,1]))
    # print(spearmanr(np.array(all_cdim)[:,0], np.array(all_cdim)[:,1]))
    # print(spearmanr(np.array(all_lyap)[:, 0], np.array(all_lyap)[:, 1]))
    print("\n")



parrot
KL: 0.21940538073230026, 0.08420726556599702
PearsonRResult(statistic=np.float64(0.6256556137850083), pvalue=np.float64(0.03950333930724558))
PearsonRResult(statistic=np.float64(-0.2437180227516353), pvalue=np.float64(0.4701802598925388))


Chronos
KL: 0.012944738791576576, 0.004009480562136341
PearsonRResult(statistic=np.float64(-0.9527973736950867), pvalue=np.float64(0.04720262630491323))


IndexError: too many indices for array: array is 1-dimensional, but 2 were indexed

In [ ]:
# Parrot
KL: 0.4088481413000285, 1.6287799243572592
PearsonRResult(statistic=np.float64(0.7229814973970862), pvalue=np.float64(1.8433805897964297e-22))
PearsonRResult(statistic=np.float64(nan), pvalue=np.float64(nan))
SignificanceResult(statistic=np.float64(0.7413761810708376), pvalue=np.float64(4.244057657913758e-24))
SignificanceResult(statistic=np.float64(0.029186974128039396), pvalue=np.float64(0.7406980941180865))

# Dynamix
KL: 0.5009202519138735, 1.649784019156996
PearsonRResult(statistic=np.float64(0.5077976864711158), pvalue=np.float64(1.1040778286610293e-09))
PearsonRResult(statistic=np.float64(0.47679227459597306), pvalue=np.float64(1.4552651924403591e-08))
SignificanceResult(statistic=np.float64(0.6143079771278589), pvalue=np.float64(1.577424077510014e-14))
SignificanceResult(statistic=np.float64(0.4788561586051743), pvalue=np.float64(1.2353937476528643e-08))

In [ ]:
print(all_cdim)
print(all_kl_dist)
print(all_lyap)

In [ ]:
from dysts.analysis import max_lyapunov_exponent_rosenstein, max_lyapunov_exponent_rosenstein_multivariate, gp_dim
from dysts.metrics import estimate_kl_divergence

import glob

results_path = "../analysis/chronos_statistics/"
context_length = 512
forecast_length = 10000 - context_length

all_lyap = list()
all_cdim = list()
all_kl_dist = list()

for trajectory in sorted(glob.glob("../data/long_trajectories/*.npy")):
    equation_name = trajectory.split("/")[-1].split(".")[0]
    print(equation_name, flush=True)
    # if equation_name in found_systems:
    #     print(f"Skipping {equation_name} because it's already in the found_systems list", flush=True)
    #     continue
    
    traj = np.load(trajectory, allow_pickle=True)
    traj = (traj - np.mean(traj, axis=0)) / np.std(traj, axis=0)

    try:

        traj_context = traj[:context_length, :]
        traj_true = traj[context_length:context_length+forecast_length, :]

        all_cdim = list() # collecting all correlation dimensions
        all_kl_dist = list() # collecting all kl divergence values


        # traj_pred = list()
        # for mode in range(traj_context.shape[1]):
        #     model = SimplexForecaster()
        #     model.fit(traj_context[:, mode])
        #     traj_pred_mode = model.forecast(forecast_length)
        #     traj_pred.append(traj_pred_mode)
        # traj_pred = np.array(traj_pred).T


        traj_pred = list()
        for mode in range(traj_context.shape[1]):
            forecast = pipeline.predict(
                inputs=torch.tensor(traj_context[:, mode]),
                prediction_length=forecast_length,
                num_samples=20,
                limit_prediction_length=False,
                )
            traj_pred_mode = np.mean(forecast[0, :, :].detach().numpy(), axis=0)
            traj_pred.append(traj_pred_mode)
        traj_pred = np.array(traj_pred).T


        
        kl_dist = estimate_kl_divergence(traj_true, traj_pred)
        if np.isinf(kl_dist): kl_dist = np.nan
        
        
        cdim_true = gp_dim(traj_true)
        cdim_pred = gp_dim(traj_pred)
        cdim = np.array([cdim_pred, cdim_true])

        lyap_true = max_lyapunov_exponent_rosenstein(traj_true)
        lyap_pred = max_lyapunov_exponent_rosenstein(traj_pred)
        lyap = np.array([lyap_pred, lyap_true])

        all_cdim.append(cdim)
        all_kl_dist.append(kl_dist)
        all_lyap.append(lyap)


    except Exception as e:
        print(e)
        print(f"Skipping {equation_name}", flush=True)
        continue
